# HyperSpy GUI toolkit comparison

This notebook gives a clean side-by-side comparison of representative `ipywidgets` and `anywidget` GUIs. It intentionally keeps `display=False` so both widgets can be shown together in the same cell.

> Modern Jupyter does **not** require `%load_ext anywidget` for this repo. Installing the package is enough.

In [ ]:
import numpy as np
import hyperspy.api as hs
import hyperspy_gui_anywidget  # noqa: F401
from IPython.display import HTML, Markdown, display

hs.preferences.GUIs.enable_traitsui_gui = False
hs.preferences.GUIs.enable_ipywidgets_gui = True
hs.preferences.GUIs.enable_anywidget_gui = True

def show_pair(title, ipy_result, aw_result):
    display(Markdown(f"### {title}"))
    display(HTML("<b>ipywidgets</b>"))
    display(ipy_result["ipywidgets"]["widget"])
    display(HTML("<b>anywidget</b>"))
    display(aw_result["anywidget"]["widget"])

def make_model():
    signal = hs.signals.Signal1D(np.ones(100))
    signal.axes_manager[0].scale = 0.1
    model = signal.create_model()
    gaussian = hs.model.components1D.Gaussian(A=10, centre=5, sigma=1)
    model.append(gaussian)
    return signal, model, gaussian

def make_smoothing_signal():
    signal = hs.signals.Signal1D(1 + np.arange(100.0) ** 2)
    signal.add_gaussian_noise(50)
    signal.change_dtype("float")
    return signal


## ROI widgets

Representative ROI widgets are a good quick check of basic layout and bidirectional syncing.

In [ ]:
show_pair(
    "SpanROI",
    hs.roi.SpanROI(left=5, right=15).gui(toolkit="ipywidgets", display=False),
    hs.roi.SpanROI(left=5, right=15).gui(toolkit="anywidget", display=False),
)

show_pair(
    "Point2DROI",
    hs.roi.Point2DROI(x=3, y=7).gui(toolkit="ipywidgets", display=False),
    hs.roi.Point2DROI(x=3, y=7).gui(toolkit="anywidget", display=False),
)

show_pair(
    "Line2DROI",
    hs.roi.Line2DROI(x1=0, y1=0, x2=10, y2=10, linewidth=2).gui(toolkit="ipywidgets", display=False),
    hs.roi.Line2DROI(x1=0, y1=0, x2=10, y2=10, linewidth=2).gui(toolkit="anywidget", display=False),
)


## Axes widgets

These examples highlight the numbered accordion titles and the navigation-slider bridge.

In [ ]:
signal_2d = hs.signals.Signal2D(np.random.random((10, 8)))
show_pair(
    "AxesManager (2D)",
    signal_2d.axes_manager.gui(toolkit="ipywidgets", display=False),
    signal_2d.axes_manager.gui(toolkit="anywidget", display=False),
)

signal_nav = hs.signals.Signal1D(np.random.random((4, 6, 20)))
show_pair(
    "Navigation sliders",
    signal_nav.axes_manager.gui_navigation_sliders(toolkit="ipywidgets", display=False),
    signal_nav.axes_manager.gui_navigation_sliders(toolkit="anywidget", display=False),
)


## Model widgets

The model path is one of the best places to compare parameter formatting and container structure.

In [ ]:
signal_1d, model, gaussian = make_model()

show_pair(
    "Parameter (Gaussian.A)",
    gaussian.A.gui(toolkit="ipywidgets", display=False),
    gaussian.A.gui(toolkit="anywidget", display=False),
)

show_pair(
    "Component (Gaussian)",
    gaussian.gui(toolkit="ipywidgets", display=False),
    gaussian.gui(toolkit="anywidget", display=False),
)

show_pair(
    "Model",
    model.gui(toolkit="ipywidgets", display=False),
    model.gui(toolkit="anywidget", display=False),
)


## Preferences and tools

Preferences and the smoothing tool are stable, widely used examples for visual comparison.

In [ ]:
show_pair(
    "Preferences",
    hs.preferences.gui(toolkit="ipywidgets", display=False),
    hs.preferences.gui(toolkit="anywidget", display=False),
)

smooth_signal = make_smoothing_signal()
show_pair(
    "Smooth Savitzky-Golay",
    smooth_signal.smooth_savitzky_golay(toolkit="ipywidgets", display=False),
    smooth_signal.smooth_savitzky_golay(toolkit="anywidget", display=False),
)


## Sync verification

The comparison is easier to trust if we also check a couple of concrete widget/object sync paths.

In [ ]:
def check(label, actual, expected):
    ok = actual == expected
    print(f"{label}: expected={expected!r}, got={actual!r} {'✓' if ok else '✗'}")
    return ok

all_ok = []

span = hs.roi.SpanROI(left=5, right=15)
span_result = span.gui(toolkit="anywidget", display=False)["anywidget"]
span_wd = span_result["wdict"]
span_wd["left"].value = -3.0
all_ok.append(check("SpanROI widget → object", span.left, -3.0))
span.right = 21.0
all_ok.append(check("SpanROI object → widget", span_wd["right"].value, 21.0))

_, _, gaussian = make_model()
param_result = gaussian.A.gui(toolkit="anywidget", display=False)["anywidget"]
param_wd = param_result["wdict"]
param_wd["value"].value = 15.0
all_ok.append(check("Gaussian.A widget → object", gaussian.A.value, 15.0))
gaussian.A.value = 3.0
all_ok.append(check("Gaussian.A object → widget", param_wd["value"].value, 3.0))

print()
print("All checks passed." if all(all_ok) else "At least one sync check failed.")
